# NFL Dynamic 2025 Prediction System - Análisis Completo

Este notebook desglosa paso a paso el sistema de predicción NFL (`team_model.py`), explicando cada componente y añadiendo análisis exploratorio de datos y del modelo.

## Estructura del notebook:
1. **Imports y configuración**
2. **Conexión a base de datos y carga de datos**
3. **Análisis exploratorio de datos (EDA)**
4. **Ingeniería de características (Feature Engineering)**
5. **Factores defensivos**
6. **Creación de datos de entrenamiento (matchups)**
7. **Entrenamiento del modelo ElasticNet**
8. **Análisis del modelo**
9. **Predicciones para la siguiente semana**
10. **Guardado en base de datos**

---
## 1. Imports y Configuración

Se utilizan las siguientes librerías:
- **SQLAlchemy / psycopg2**: conexión a PostgreSQL
- **Pandas / NumPy**: manipulación de datos
- **Scikit-learn**: modelo ElasticNet, métricas, escalado
- **Matplotlib / Seaborn**: visualización
- **python-dotenv**: variables de entorno desde `.env`

In [ ]:
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import pandas as pd
import seaborn as sns
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import RobustScaler
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

### Funciones auxiliares

Estas funciones convierten tipos numpy (int64, float64, etc.) a tipos nativos de Python para compatibilidad con PostgreSQL, que no acepta tipos numpy directamente.

In [ ]:
def convert_numpy_types(value):
    """Convertir tipos numpy a tipos nativos de Python para compatibilidad con PostgreSQL"""
    if isinstance(value, (np.integer, np.int64, np.int32)):
        return int(value)
    elif isinstance(value, (np.floating, np.float64, np.float32)):
        return float(value)
    elif isinstance(value, np.bool_):
        return bool(value)
    elif isinstance(value, np.ndarray):
        return value.tolist()
    elif pd.isna(value) or value is None:
        return None
    else:
        return value

def prepare_db_params(params_dict):
    """Preparar parámetros para la base de datos convirtiendo tipos numpy"""
    return {key: convert_numpy_types(value) for key, value in params_dict.items()}

---
## 2. Conexión a Base de Datos y Carga de Datos

El sistema carga tres tablas principales desde PostgreSQL:
- **nfl_team_stats**: estadísticas de equipo por semana (yards, puntos, turnovers, etc.)
- **rankings**: rankings defensivos/ofensivos semanales
- **nfl_schedules**: calendario de juegos con resultados

Solo se cargan datos desde la temporada 2023 en adelante.

In [ ]:
load_dotenv()

DB_NAME = os.getenv("DB_NAME")
DB_HOST = os.getenv("DB_HOST")
DB_PASS = os.getenv("DB_PASS")
DB_PORT = os.getenv("DB_PORT", "5433")
DB_USER = os.getenv("DB_USER")

engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    connect_args={'connect_timeout': 10}
)

with engine.connect() as conn:
    dfT = pd.read_sql(text("SELECT * FROM nfl_team_stats WHERE season >= 2023"), conn)
    dfR = pd.read_sql(text("SELECT * FROM rankings WHERE season >= 2023"), conn)
    dfS = pd.read_sql(text("SELECT * FROM nfl_schedules WHERE season >= 2023"), conn)

current_season = 2025
current_season_data = dfT[dfT['season'] == current_season]
max_available_week = int(current_season_data['week'].max()) if not current_season_data.empty else 0

print(f"Datos cargados:")
print(f"  nfl_team_stats: {dfT.shape[0]} registros, {dfT.shape[1]} columnas")
print(f"  rankings: {dfR.shape[0]} registros, {dfR.shape[1]} columnas")
print(f"  nfl_schedules: {dfS.shape[0]} registros, {dfS.shape[1]} columnas")
print(f"  Temporadas disponibles: {sorted(dfT['season'].unique())}")
print(f"  Última semana con datos en {current_season}: {max_available_week}")

---
## 3. Análisis Exploratorio de Datos (EDA)

Exploramos la estructura y distribución de los datos antes de construir el modelo.

### 3.1 Estructura de nfl_team_stats

In [ ]:
print("Columnas de nfl_team_stats:")
print(dfT.columns.tolist())
print(f"\nPrimeras filas:")
dfT.head()

In [ ]:
dfT.describe()

In [ ]:
print("Valores nulos por columna:")
null_counts = dfT.isnull().sum()
print(null_counts[null_counts > 0])

### 3.2 Distribución de puntos por temporada

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de puntos por temporada
for season in sorted(dfT['season'].unique()):
    season_data = dfT[dfT['season'] == season]['points']
    axes[0].hist(season_data, alpha=0.5, label=f'{season}', bins=20)
axes[0].set_xlabel('Puntos')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Puntos por Temporada')
axes[0].legend()

# Boxplot de puntos por temporada
dfT.boxplot(column='points', by='season', ax=axes[1])
axes[1].set_title('Puntos por Temporada')
axes[1].set_xlabel('Temporada')
axes[1].set_ylabel('Puntos')

plt.tight_layout()
plt.show()

### 3.3 Distribución de yardas (total, pase, carrera)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['total_yards', 'passing_yards', 'rushing_yards']
titles = ['Yardas Totales', 'Yardas de Pase', 'Yardas de Carrera']

for ax, metric, title in zip(axes, metrics, titles):
    if metric in dfT.columns:
        sns.histplot(data=dfT, x=metric, hue='season', ax=ax, kde=True, alpha=0.4)
        ax.set_title(title)
        ax.set_xlabel(title)

plt.tight_layout()
plt.show()

### 3.4 Correlación entre métricas ofensivas

In [ ]:
offensive_cols = ['points', 'total_yards', 'passing_yards', 'rushing_yards', 'turnovers']
available_cols = [c for c in offensive_cols if c in dfT.columns]

fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = dfT[available_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, ax=ax, fmt='.2f')
ax.set_title('Correlación entre Métricas Ofensivas')
plt.tight_layout()
plt.show()

### 3.5 Promedios por equipo (temporada más reciente)

In [ ]:
latest_season = dfT['season'].max()
latest_data = dfT[dfT['season'] == latest_season]

team_avgs = latest_data.groupby('team')[['points', 'total_yards', 'passing_yards', 'rushing_yards']].mean()
team_avgs = team_avgs.sort_values('points', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
team_avgs['points'].plot(kind='bar', ax=ax, color='steelblue')
ax.set_title(f'Promedio de Puntos por Equipo - Temporada {latest_season}')
ax.set_xlabel('Equipo')
ax.set_ylabel('Puntos promedio')
ax.axhline(y=team_avgs['points'].mean(), color='red', linestyle='--', label='Media liga')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 3.6 Análisis de la ventaja de local

In [ ]:
games_with_scores = dfS[(dfS['away_score'].notna()) & (dfS['home_score'].notna())].copy()
games_with_scores['home_win'] = games_with_scores['home_score'] > games_with_scores['away_score']
games_with_scores['point_diff'] = games_with_scores['home_score'] - games_with_scores['away_score']

home_advantage = games_with_scores.groupby('season').agg(
    home_win_pct=('home_win', 'mean'),
    avg_point_diff=('point_diff', 'mean'),
    total_games=('home_win', 'count')
).round(3)

print("Ventaja de local por temporada:")
print(home_advantage)
print(f"\nVentaja promedio de local (puntos): {games_with_scores['point_diff'].mean():.2f}")
print(f"% victorias local: {games_with_scores['home_win'].mean():.1%}")

### 3.7 Estructura de rankings

In [ ]:
print("Columnas de rankings:")
print(dfR.columns.tolist())
dfR.head()

---
## 4. Ingeniería de Características (Feature Engineering)

Se crean más de 100 características a partir de las estadísticas base. Las transformaciones principales son:

1. **Promedios móviles** (ventanas de 3, 4 y 6 semanas): capturan rendimiento reciente
2. **Desviaciones estándar móviles**: capturan consistencia/variabilidad
3. **Tendencias** (diferencia entre promedios consecutivos): capturan si el equipo mejora o empeora
4. **Eficiencia ofensiva**: puntos por yarda total (mide eficiencia en zona roja)
5. **Ratio pase/carrera**: estilo de juego del equipo
6. **Promedio reciente de puntos** (ventana de 2): forma muy reciente

In [ ]:
dfT_features = dfT.sort_values(['team', 'season', 'week']).copy()

base_metrics = ['total_yards', 'passing_yards', 'rushing_yards', 'points', 
               'turnovers', 'op_total_yards', 'op_passing_yards', 
               'op_rushing_yards', 'op_points']

windows = [3, 4, 6]

for window in windows:
    for metric in base_metrics:
        if metric not in dfT_features.columns:
            continue
        # Promedio móvil
        dfT_features[f'{metric}_avg_{window}'] = (
            dfT_features.groupby(['team', 'season'])[metric]
            .rolling(window=window, min_periods=1)
            .mean()
            .reset_index(level=[0, 1], drop=True)
        )
        # Desviación estándar móvil
        dfT_features[f'{metric}_std_{window}'] = (
            dfT_features.groupby(['team', 'season'])[metric]
            .rolling(window=window, min_periods=1)
            .std()
            .reset_index(level=[0, 1], drop=True)
        ).fillna(0)
        # Tendencia
        dfT_features[f'{metric}_trend_{window}'] = (
            dfT_features.groupby(['team', 'season'])[f'{metric}_avg_{window}']
            .diff(1)
            .fillna(0)
        )

# Eficiencia ofensiva: puntos por yarda
dfT_features['offensive_efficiency'] = (
    dfT_features['points'] / (dfT_features['total_yards'] + 1)
)

# Ratio pase/carrera
dfT_features['pass_rush_ratio'] = (
    dfT_features['passing_yards'] / (dfT_features['rushing_yards'] + 1)
)

# Promedio reciente de puntos (2 semanas)
dfT_features['recent_points_avg'] = (
    dfT_features.groupby(['team', 'season'])['points']
    .rolling(window=2, min_periods=1)
    .mean()
    .reset_index(level=[0, 1], drop=True)
)

print(f"Total de columnas después del feature engineering: {len(dfT_features.columns)}")
print(f"Columnas nuevas creadas: {len(dfT_features.columns) - len(dfT.columns)}")

### 4.1 Visualización de features generados

Veamos cómo se ven los promedios móviles para un equipo ejemplo.

In [ ]:
# Seleccionar un equipo ejemplo para visualizar
example_team = dfT_features['team'].value_counts().index[0]  # equipo con más datos
team_data = dfT_features[(dfT_features['team'] == example_team) & (dfT_features['season'] == 2024)].copy()

if not team_data.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Puntos: valor real vs promedios móviles
    axes[0, 0].plot(team_data['week'], team_data['points'], 'o-', label='Real', alpha=0.7)
    axes[0, 0].plot(team_data['week'], team_data['points_avg_3'], '--', label='Avg 3 sem')
    axes[0, 0].plot(team_data['week'], team_data['points_avg_6'], '--', label='Avg 6 sem')
    axes[0, 0].set_title(f'{example_team} - Puntos 2024')
    axes[0, 0].legend()
    axes[0, 0].set_xlabel('Semana')
    
    # Yardas totales
    axes[0, 1].plot(team_data['week'], team_data['total_yards'], 'o-', label='Real', alpha=0.7)
    axes[0, 1].plot(team_data['week'], team_data['total_yards_avg_4'], '--', label='Avg 4 sem')
    axes[0, 1].set_title(f'{example_team} - Yardas Totales 2024')
    axes[0, 1].legend()
    axes[0, 1].set_xlabel('Semana')
    
    # Tendencia de puntos
    axes[1, 0].bar(team_data['week'], team_data['points_trend_4'], 
                   color=['green' if x > 0 else 'red' for x in team_data['points_trend_4']])
    axes[1, 0].set_title(f'{example_team} - Tendencia Puntos (4 sem)')
    axes[1, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    axes[1, 0].set_xlabel('Semana')
    
    # Eficiencia ofensiva
    axes[1, 1].plot(team_data['week'], team_data['offensive_efficiency'], 'o-', color='purple')
    axes[1, 1].set_title(f'{example_team} - Eficiencia Ofensiva 2024')
    axes[1, 1].set_xlabel('Semana')
    axes[1, 1].set_ylabel('Puntos / Yarda')
    
    plt.suptitle(f'Feature Engineering - {example_team} Temporada 2024', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

---
## 5. Factores Defensivos

Los rankings defensivos se transforman en "factores" entre 0 y 1:
- Factor cercano a 1 = defensa élite (bajo ranking numérico)
- Factor cercano a 0 = defensa débil (alto ranking numérico)

La fórmula usa una exponencial (^0.8) para suavizar las diferencias entre rankings medios.

In [ ]:
def rank_to_factor(rank, total_teams=32):
    if pd.isna(rank):
        return 0.5
    normalized = (total_teams - rank) / (total_teams - 1)
    return normalized ** 0.8

rank_to_factor_map = {
    'op_total_yards_rank': 'op_total_yards_factor',
    'op_points_rank': 'op_points_factor'
}

for rank_col, factor_col in rank_to_factor_map.items():
    if rank_col in dfR.columns:
        dfR[factor_col] = dfR[rank_col].apply(rank_to_factor)

# Visualizar la transformación
ranks = np.arange(1, 33)
factors = [rank_to_factor(r) for r in ranks]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ranks, factors, 'o-')
ax.set_xlabel('Ranking Defensivo (1=mejor, 32=peor)')
ax.set_ylabel('Factor Defensivo')
ax.set_title('Transformación de Ranking a Factor Defensivo')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Factor neutro (0.5)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Creación de Datos de Entrenamiento (Matchups)

Para entrenar el modelo, se crean pares de enfrentamiento a partir de juegos históricos (2023-2024):

- Para cada juego del calendario con resultado conocido:
  1. Se obtienen las estadísticas del equipo **ANTES** del juego (rolling stats hasta week-1)
  2. Se obtienen los rankings defensivos del oponente
  3. Se registra el resultado real como target
  
- Cada juego genera **2 registros**: uno para el equipo visitante y otro para el local
- La variable `is_home` indica si es local (1) o visitante (0)

In [ ]:
# Calcular estadísticas finales de 2024 (para fallback en predicciones semana 1)
team_2024_stats = {}
teams_2024 = dfT_features[dfT_features['season'] == 2024]['team'].unique()

for team in teams_2024:
    team_data = dfT_features[
        (dfT_features['team'] == team) & 
        (dfT_features['season'] == 2024)
    ].sort_values('week')
    if not team_data.empty:
        team_2024_stats[team] = team_data.tail(1).iloc[0]

print(f"Estadísticas de 2024 calculadas para {len(team_2024_stats)} equipos")

In [ ]:
matchups = []

training_games = dfS[
    (dfS['season'] >= 2023) &
    (pd.notna(dfS['away_score'])) & 
    (pd.notna(dfS['home_score']))
]

print(f"Juegos de entrenamiento disponibles: {len(training_games)}")

for _, game in training_games.iterrows():
    away_stats = dfT_features[
        (dfT_features['team'] == game['away_team']) & 
        (dfT_features['season'] == game['season']) & 
        (dfT_features['week'] < game['week'])
    ].tail(1)
    
    home_stats = dfT_features[
        (dfT_features['team'] == game['home_team']) & 
        (dfT_features['season'] == game['season']) & 
        (dfT_features['week'] < game['week'])
    ].tail(1)
    
    # Fallback: si no hay datos previos, usar la misma semana
    if away_stats.empty:
        away_stats = dfT_features[
            (dfT_features['team'] == game['away_team']) & 
            (dfT_features['season'] == game['season']) & 
            (dfT_features['week'] == game['week'])
        ].head(1)
    
    if home_stats.empty:
        home_stats = dfT_features[
            (dfT_features['team'] == game['home_team']) & 
            (dfT_features['season'] == game['season']) & 
            (dfT_features['week'] == game['week'])
        ].head(1)
    
    if away_stats.empty or home_stats.empty:
        continue
    
    away_def = dfR[
        (dfR['team'] == game['away_team']) & 
        (dfR['season'] == game['season']) & 
        (dfR['week'] <= game['week'])
    ].tail(1)
    
    home_def = dfR[
        (dfR['team'] == game['home_team']) & 
        (dfR['season'] == game['season']) & 
        (dfR['week'] <= game['week'])
    ].tail(1)
    
    if away_def.empty or home_def.empty:
        continue
    
    away_actual = dfT_features[
        (dfT_features['team'] == game['away_team']) & 
        (dfT_features['season'] == game['season']) & 
        (dfT_features['week'] == game['week'])
    ]
    
    home_actual = dfT_features[
        (dfT_features['team'] == game['home_team']) & 
        (dfT_features['season'] == game['season']) & 
        (dfT_features['week'] == game['week'])
    ]
    
    if away_actual.empty or home_actual.empty:
        continue
    
    for team_type in ['away', 'home']:
        is_home = 1 if team_type == 'home' else 0
        
        if team_type == 'away':
            team_stats = away_stats.iloc[0]
            opp_def = home_def.iloc[0]
            actual_stats = away_actual.iloc[0]
            actual_points = game['away_score']
        else:
            team_stats = home_stats.iloc[0]
            opp_def = away_def.iloc[0]
            actual_stats = home_actual.iloc[0]
            actual_points = game['home_score']
        
        matchup = {
            'season': game['season'],
            'week': game['week'],
            'is_home': is_home,
            'team_points_avg_3': team_stats.get('points_avg_3', 24),
            'team_points_avg_4': team_stats.get('points_avg_4', 24),
            'team_points_avg_6': team_stats.get('points_avg_6', 24),
            'team_total_yards_avg_3': team_stats.get('total_yards_avg_3', 350),
            'team_total_yards_avg_4': team_stats.get('total_yards_avg_4', 350),
            'team_total_yards_avg_6': team_stats.get('total_yards_avg_6', 350),
            'team_passing_yards_avg_4': team_stats.get('passing_yards_avg_4', 230),
            'team_rushing_yards_avg_4': team_stats.get('rushing_yards_avg_4', 120),
            'team_points_std_4': team_stats.get('points_std_4', 7),
            'team_total_yards_std_4': team_stats.get('total_yards_std_4', 50),
            'team_points_trend_4': team_stats.get('points_trend_4', 0),
            'team_total_yards_trend_4': team_stats.get('total_yards_trend_4', 0),
            'team_off_efficiency': team_stats.get('offensive_efficiency', 0.07),
            'team_pass_rush_ratio': team_stats.get('pass_rush_ratio', 2.0),
            'team_recent_points': team_stats.get('recent_points_avg', 24),
            'opp_def_total_rank': opp_def.get('op_total_yards_rank', 16),
            'opp_def_passing_rank': opp_def.get('op_passing_yards_rank', 16),
            'opp_def_rushing_rank': opp_def.get('op_rushing_yards_rank', 16),
            'opp_def_points_rank': opp_def.get('op_points_rank', 16),
            'opp_def_total_factor': opp_def.get('op_total_yards_factor', 0.5),
            'opp_def_points_factor': opp_def.get('op_points_factor', 0.5),
            'actual_total_yards': actual_stats.get('total_yards', 350),
            'actual_passing_yards': actual_stats.get('passing_yards', 230),
            'actual_rushing_yards': actual_stats.get('rushing_yards', 120),
            'actual_points': actual_points
        }
        
        matchups.append(matchup)

matchup_data = pd.DataFrame(matchups)
print(f"\nDatos de entrenamiento creados: {len(matchup_data)} registros")
print(f"Columnas: {len(matchup_data.columns)}")

### 6.1 Análisis de los datos de entrenamiento

In [ ]:
matchup_data.describe()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

targets = ['actual_points', 'actual_total_yards', 'actual_passing_yards', 'actual_rushing_yards']
titles = ['Puntos Reales', 'Yardas Totales Reales', 'Yardas Pase Reales', 'Yardas Carrera Reales']

for ax, target, title in zip(axes.flat, targets, titles):
    sns.histplot(data=matchup_data, x=target, hue='is_home', ax=ax, kde=True, alpha=0.4)
    ax.set_title(title)
    ax.legend(['Visitante', 'Local'])

plt.suptitle('Distribución de Variables Target por Localía', fontsize=14)
plt.tight_layout()
plt.show()

---
## 7. Entrenamiento del Modelo ElasticNet

Se entrena un modelo **ElasticNet** por cada métrica a predecir:
- `points`: puntos anotados
- `total_yards`: yardas totales
- `passing_yards`: yardas de pase
- `rushing_yards`: yardas de carrera

**¿Por qué ElasticNet?**
- Combina regularización L1 (Lasso) y L2 (Ridge)
- `alpha=0.1`: fuerza de regularización moderada
- `l1_ratio=0.5`: balance entre L1 y L2
- Maneja bien la multicolinealidad entre features (promedios de diferentes ventanas)
- Más robusto que regresión lineal simple con muchos features correlacionados

**¿Por qué RobustScaler?**
- Usa mediana e IQR en lugar de media y desviación estándar
- Menos sensible a outliers (juegos atípicos con muchos/pocos puntos)

In [ ]:
metrics = {
    'total_yards': 'actual_total_yards',
    'passing_yards': 'actual_passing_yards', 
    'rushing_yards': 'actual_rushing_yards',
    'points': 'actual_points'
}

feature_cols = [
    'season', 'week', 'is_home',
    'team_points_avg_3', 'team_points_avg_4', 'team_points_avg_6',
    'team_total_yards_avg_3', 'team_total_yards_avg_4', 'team_total_yards_avg_6',
    'team_passing_yards_avg_4', 'team_rushing_yards_avg_4',
    'team_points_std_4', 'team_total_yards_std_4',
    'team_points_trend_4', 'team_total_yards_trend_4',
    'team_off_efficiency', 'team_pass_rush_ratio', 'team_recent_points',
    'opp_def_total_rank', 'opp_def_passing_rank', 
    'opp_def_rushing_rank', 'opp_def_points_rank',
    'opp_def_total_factor', 'opp_def_points_factor'
]

models = {}
scalers = {}
training_results = {}
test_predictions = {}  # Para análisis posterior

print(f"Features utilizados: {len(feature_cols)}")
print(f"Registros de entrenamiento: {len(matchup_data)}")
print()

for metric_name, target_col in metrics.items():
    print(f"Entrenando modelo para: {metric_name}")
    print("-" * 40)
    
    df_clean = matchup_data.dropna(subset=[target_col] + feature_cols)
    X = df_clean[feature_cols]
    y = df_clean[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=df_clean['is_home']
    )
    
    scaler = RobustScaler()
    model = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42)
    
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    models[metric_name] = model
    scalers[metric_name] = scaler
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    training_results[metric_name] = {
        'mae': mae, 'rmse': rmse, 'r2': r2,
        'train_size': len(X_train), 'test_size': len(X_test)
    }
    
    test_predictions[metric_name] = {'y_test': y_test, 'y_pred': y_pred}
    
    print(f"  Train: {len(X_train)} | Test: {len(X_test)}")
    print(f"  MAE:  {mae:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R²:   {r2:.3f}")
    print()

---
## 8. Análisis del Modelo

### 8.1 Resumen de métricas

In [ ]:
results_df = pd.DataFrame(training_results).T
results_df.index.name = 'Métrica'
print("Resumen de rendimiento del modelo:")
results_df

### 8.2 Predicciones vs Valores Reales

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ax, (metric_name, data) in zip(axes.flat, test_predictions.items()):
    y_test = data['y_test']
    y_pred = data['y_pred']
    
    ax.scatter(y_test, y_pred, alpha=0.3, s=20)
    
    # Línea perfecta
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Predicción perfecta')
    
    ax.set_xlabel('Valor Real')
    ax.set_ylabel('Predicción')
    ax.set_title(f'{metric_name} (R²={training_results[metric_name]["r2"]:.3f})')
    ax.legend()

plt.suptitle('Predicciones vs Valores Reales - ElasticNet', fontsize=14)
plt.tight_layout()
plt.show()

### 8.3 Distribución de Errores (Residuos)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (metric_name, data) in zip(axes.flat, test_predictions.items()):
    residuals = data['y_test'].values - data['y_pred']
    
    sns.histplot(residuals, kde=True, ax=ax, color='steelblue')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5)
    ax.set_title(f'{metric_name} - Residuos (μ={residuals.mean():.2f}, σ={residuals.std():.2f})')
    ax.set_xlabel('Error (Real - Predicción)')

plt.suptitle('Distribución de Errores por Métrica', fontsize=14)
plt.tight_layout()
plt.show()

### 8.4 Importancia de Features (Coeficientes ElasticNet)

ElasticNet asigna coeficientes a cada feature. Los coeficientes más grandes (en valor absoluto) indican mayor influencia. Coeficientes en 0 indican features eliminados por la regularización L1.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 16))

for ax, (metric_name, model) in zip(axes.flat, models.items()):
    coefs = pd.Series(model.coef_, index=feature_cols)
    coefs_sorted = coefs.reindex(coefs.abs().sort_values(ascending=True).index)
    
    colors = ['green' if c > 0 else 'red' for c in coefs_sorted]
    coefs_sorted.plot(kind='barh', ax=ax, color=colors)
    ax.set_title(f'{metric_name} - Coeficientes')
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.set_xlabel('Coeficiente')

plt.suptitle('Coeficientes del Modelo ElasticNet (verde=positivo, rojo=negativo)', fontsize=14)
plt.tight_layout()
plt.show()

### 8.5 Features con coeficiente cero (eliminados por L1)

In [ ]:
print("Features eliminados por regularización L1 (coeficiente = 0):")
print("=" * 50)
for metric_name, model in models.items():
    zero_coefs = [f for f, c in zip(feature_cols, model.coef_) if abs(c) < 1e-10]
    non_zero = len(feature_cols) - len(zero_coefs)
    print(f"\n{metric_name}: {non_zero}/{len(feature_cols)} features activos")
    if zero_coefs:
        print(f"  Eliminados: {zero_coefs}")

### 8.6 Análisis de error por localía

In [ ]:
# Análisis del error separado por local/visitante
for metric_name, target_col in metrics.items():
    df_clean = matchup_data.dropna(subset=[target_col] + feature_cols)
    X = df_clean[feature_cols]
    y = df_clean[target_col]
    
    X_scaled = scalers[metric_name].transform(X)
    y_pred_all = models[metric_name].predict(X_scaled)
    
    errors = pd.DataFrame({
        'error': np.abs(y.values - y_pred_all),
        'is_home': df_clean['is_home'].values
    })
    
    home_mae = errors[errors['is_home'] == 1]['error'].mean()
    away_mae = errors[errors['is_home'] == 0]['error'].mean()
    print(f"{metric_name:15s} | MAE Local: {home_mae:.2f} | MAE Visitante: {away_mae:.2f} | Diff: {abs(home_mae-away_mae):.2f}")

---
## 9. Predicciones para la Siguiente Semana

El sistema predice la siguiente semana disponible:
- Si `max_available_week = N`, predice semana `N+1`
- Si no hay datos de 2025, predice semana 1 usando estadísticas finales de 2024

Para cada juego:
1. Obtiene estadísticas recientes de cada equipo
2. Obtiene rankings defensivos del oponente
3. Predice con el modelo entrenado
4. Aplica ajustes defensivos y de eficiencia
5. Suma ventaja de local (+2.8 pts, +8 yds para el equipo local)

In [ ]:
def get_team_current_stats(team, target_week, season=2025):
    """Obtener estadísticas actuales del equipo para predicción"""
    if target_week == 1:
        if team in team_2024_stats:
            return team_2024_stats[team]
        return None
    else:
        team_stats_2025 = dfT_features[
            (dfT_features['team'] == team) & 
            (dfT_features['season'] == season) & 
            (dfT_features['week'] < target_week)
        ].tail(1)
        
        if not team_stats_2025.empty:
            return team_stats_2025.iloc[0]
        elif team in team_2024_stats:
            return team_2024_stats[team]
        return None

def get_team_defensive_stats(team, target_week, season=2025):
    """Obtener estadísticas defensivas del equipo"""
    if target_week == 1:
        def_stats = dfR[(dfR['team'] == team) & (dfR['season'] == 2024)].tail(1)
    else:
        def_stats = dfR[
            (dfR['team'] == team) & (dfR['season'] == season) & (dfR['week'] < target_week)
        ].tail(1)
        if def_stats.empty:
            def_stats = dfR[(dfR['team'] == team) & (dfR['season'] == 2024)].tail(1)
    return def_stats if not def_stats.empty else None


def predict_game(away_team, home_team, week, season=2025):
    """Predecir un juego individual"""
    away_stats = get_team_current_stats(away_team, week, season)
    home_stats = get_team_current_stats(home_team, week, season)
    
    if away_stats is None or home_stats is None:
        return None
    
    away_def = get_team_defensive_stats(away_team, week, season)
    home_def = get_team_defensive_stats(home_team, week, season)
    
    if away_def is None or home_def is None:
        return None
    
    def prepare_features(team_stats, opp_def, is_home):
        stats = team_stats
        def_data = opp_def.iloc[0] if hasattr(opp_def, 'iloc') else opp_def
        return {
            'season': season, 'week': week, 'is_home': is_home,
            'team_points_avg_3': stats.get('points_avg_3', 24),
            'team_points_avg_4': stats.get('points_avg_4', 24),
            'team_points_avg_6': stats.get('points_avg_6', 24),
            'team_total_yards_avg_3': stats.get('total_yards_avg_3', 350),
            'team_total_yards_avg_4': stats.get('total_yards_avg_4', 350),
            'team_total_yards_avg_6': stats.get('total_yards_avg_6', 350),
            'team_passing_yards_avg_4': stats.get('passing_yards_avg_4', 230),
            'team_rushing_yards_avg_4': stats.get('rushing_yards_avg_4', 120),
            'team_points_std_4': stats.get('points_std_4', 7),
            'team_total_yards_std_4': stats.get('total_yards_std_4', 50),
            'team_points_trend_4': stats.get('points_trend_4', 0),
            'team_total_yards_trend_4': stats.get('total_yards_trend_4', 0),
            'team_off_efficiency': stats.get('offensive_efficiency', 0.07),
            'team_pass_rush_ratio': stats.get('pass_rush_ratio', 2.0),
            'team_recent_points': stats.get('recent_points_avg', 24),
            'opp_def_total_rank': def_data.get('op_total_yards_rank', 16),
            'opp_def_passing_rank': def_data.get('op_passing_yards_rank', 16),
            'opp_def_rushing_rank': def_data.get('op_rushing_yards_rank', 16),
            'opp_def_points_rank': def_data.get('op_points_rank', 16),
            'opp_def_total_factor': def_data.get('op_total_yards_factor', 0.5),
            'opp_def_points_factor': def_data.get('op_points_factor', 0.5)
        }
    
    away_features = prepare_features(away_stats, home_def, 0)
    home_features = prepare_features(home_stats, away_def, 1)
    
    pred_metrics = ['total_yards', 'passing_yards', 'rushing_yards', 'points']
    away_predictions = {}
    home_predictions = {}
    
    for metric in pred_metrics:
        if metric not in models:
            continue
        
        model = models[metric]
        scaler = scalers[metric]
        adjustment_factors = {'total_yards': 0.18, 'passing_yards': 0.22, 'rushing_yards': 0.20, 'points': 0.15}
        factor = adjustment_factors.get(metric, 0.15)
        
        # Visitante
        away_X = pd.DataFrame([away_features])[feature_cols]
        away_pred = model.predict(scaler.transform(away_X))[0]
        def_factor = away_features['opp_def_total_factor'] if metric != 'points' else away_features['opp_def_points_factor']
        if metric == 'total_yards':
            def_factor = away_features['opp_def_total_factor']
        def_adjustment = (def_factor - 0.5) * factor
        efficiency_adjustment = (away_features['team_off_efficiency'] - 0.07) * 0.1
        away_predictions[metric] = max(0, away_pred + away_pred * (def_adjustment + efficiency_adjustment))
        
        # Local
        home_X = pd.DataFrame([home_features])[feature_cols]
        home_pred = model.predict(scaler.transform(home_X))[0]
        def_factor = home_features['opp_def_total_factor'] if metric != 'points' else home_features['opp_def_points_factor']
        if metric == 'total_yards':
            def_factor = home_features['opp_def_total_factor']
        def_adjustment = (def_factor - 0.5) * factor
        efficiency_adjustment = (home_features['team_off_efficiency'] - 0.07) * 0.1
        home_final = home_pred + home_pred * (def_adjustment + efficiency_adjustment)
        
        # Ventaja de local
        if metric == 'points':
            home_final += 2.8
        elif metric == 'total_yards':
            home_final += 8
        
        home_predictions[metric] = max(0, home_final)
    
    away_points = away_predictions.get('points', 0)
    home_points = home_predictions.get('points', 0)
    winner = 'home' if home_points > away_points else 'away'
    point_diff = abs(home_points - away_points)
    confidence = min(point_diff / 16.0, 1.0)
    
    return {
        'away_team': away_team, 'home_team': home_team,
        'week': week, 'season': season,
        'away_predictions': away_predictions,
        'home_predictions': home_predictions,
        'predicted_winner': winner,
        'point_differential': point_diff,
        'confidence': confidence
    }

### 9.1 Generar predicciones para la siguiente semana

In [ ]:
next_week = max_available_week + 1 if max_available_week > 0 else 1

week_games = dfS[
    (dfS['season'] == current_season) & 
    (dfS['week'] == next_week)
]

print(f"PREDICCIONES SEMANA {next_week} - TEMPORADA {current_season}")
print("=" * 70)
print(f"Juegos encontrados: {len(week_games)}")
print()

predictions = []
for _, game in week_games.iterrows():
    pred = predict_game(game['away_team'], game['home_team'], next_week, current_season)
    if pred:
        predictions.append(pred)
        winner_team = pred['home_team'] if pred['predicted_winner'] == 'home' else pred['away_team']
        print(f"{pred['away_team']:>4s} @ {pred['home_team']:<4s} | "
              f"Score: {pred['away_predictions']['points']:.1f} - {pred['home_predictions']['points']:.1f} | "
              f"Ganador: {winner_team} | Confianza: {pred['confidence']:.0%}")

print(f"\nTotal predicciones generadas: {len(predictions)}")

### 9.2 Visualización de predicciones

In [ ]:
if predictions:
    pred_df = pd.DataFrame([{
        'matchup': f"{p['away_team']} @ {p['home_team']}",
        'away_pts': p['away_predictions']['points'],
        'home_pts': p['home_predictions']['points'],
        'confidence': p['confidence'],
        'winner': p['home_team'] if p['predicted_winner'] == 'home' else p['away_team']
    } for p in predictions])
    
    fig, axes = plt.subplots(1, 2, figsize=(16, max(6, len(predictions) * 0.4)))
    
    # Gráfico de puntos predichos
    y_pos = range(len(pred_df))
    axes[0].barh(y_pos, pred_df['home_pts'], alpha=0.7, label='Local', color='steelblue')
    axes[0].barh(y_pos, -pred_df['away_pts'], alpha=0.7, label='Visitante', color='coral')
    axes[0].set_yticks(y_pos)
    axes[0].set_yticklabels(pred_df['matchup'])
    axes[0].set_xlabel('Puntos Predichos')
    axes[0].set_title(f'Predicciones de Puntos - Semana {next_week}')
    axes[0].legend()
    axes[0].axvline(x=0, color='black', linewidth=0.5)
    
    # Gráfico de confianza
    colors = ['green' if c > 0.5 else 'orange' if c > 0.3 else 'red' for c in pred_df['confidence']]
    axes[1].barh(y_pos, pred_df['confidence'], color=colors, alpha=0.7)
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(pred_df['matchup'])
    axes[1].set_xlabel('Confianza')
    axes[1].set_title('Nivel de Confianza por Juego')
    axes[1].set_xlim(0, 1)
    axes[1].axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

---
## 10. Guardado en Base de Datos

El sistema guarda las predicciones en la tabla `game_predictions` con la siguiente lógica:
- Si el `game_id` ya existe: actualiza solo las predicciones (no toca resultados reales)
- Si no existe: inserta un nuevo registro

Después de cada semana jugada, el sistema:
1. Actualiza los resultados reales en `game_predictions`
2. Calcula métricas de accuracy en `week_game_results`

El `game_id` tiene formato: `{season}_W{week}_{away_team}_{home_team}`

In [ ]:
# Ejemplo de game_id y estructura de datos guardados
if predictions:
    example = predictions[0]
    game_id = f"{example['season']}_W{example['week']}_{example['away_team']}_{example['home_team']}"
    
    print(f"Ejemplo de game_id: {game_id}")
    print(f"\nDatos que se guardan:")
    print(f"  Predicciones visitante: {example['away_predictions']}")
    print(f"  Predicciones local: {example['home_predictions']}")
    print(f"  Ganador predicho: {example['predicted_winner']}")
    print(f"  Diferencial: {example['point_differential']:.2f}")
    print(f"  Confianza: {example['confidence']:.3f}")

In [ ]:
# Descomentar para guardar en base de datos:

# def save_predictions(predictions, engine):
#     with engine.connect() as conn:
#         for pred in predictions:
#             game_id = f"{pred['season']}_W{pred['week']}_{pred['away_team']}_{pred['home_team']}"
#             check = conn.execute(text("SELECT id FROM game_predictions WHERE game_id = :gid"), {"gid": game_id}).fetchone()
#             if check:
#                 print(f"  Ya existe: {game_id} - actualizando")
#             else:
#                 print(f"  Nuevo: {game_id}")
#             # ... (lógica completa en team_model.py)
#         conn.commit()
#
# save_predictions(predictions, engine)

print("Para guardar predicciones, descomentar el código anterior o ejecutar:")
print("  python team_model.py")

---
## Resumen

| Componente | Descripción |
|---|---|
| **Modelo** | ElasticNet (alpha=0.1, l1_ratio=0.5) |
| **Scaler** | RobustScaler (robusto a outliers) |
| **Features** | 24 variables: promedios móviles, tendencias, eficiencia, rankings defensivos |
| **Targets** | 4 métricas: puntos, yardas totales, yardas pase, yardas carrera |
| **Datos de entrenamiento** | 2023-2024 (2 registros por juego: local y visitante) |
| **Ajustes post-modelo** | Factores defensivos del oponente + ventaja de local |
| **Ventaja local** | +2.8 puntos, +8 yardas totales |
| **Confianza** | min(diferencial_puntos / 16, 1.0) |